# Lezione 28 — Retrieval ibrido: combinare tutti i segnali

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezione 18
(similarita' coseno), Lezione 21 (metriche di retrieval), Lezione 25
(importanza composita), Lezione 27 (grafo). Oggi tutti e quattro
convergono in un'unica funzione di ranking, confrontata onestamente con
il solo embedding — **senza aspettarsi che vincere su ogni metrica sia
scontato**.

**Cosa saprai fare alla fine:** costruire un punteggio di retrieval
ibrido che combina similarita' per embedding, segnale di grafo e
importanza composita, misurarlo con le metriche della Lezione 21, e
leggere onestamente un risultato **misto** invece di uno a senso unico.

## Parte 1 — Teoria: perche' combinare, e cosa aspettarsi

Quattro lezioni hanno costruito quattro segnali diversi per la stessa
coppia (query, candidato):

- **similarita' per embedding** (Lezione 18) — quanto i due testi si
  somigliano nello spazio appreso;
- **segnale di grafo** (Lezione 27) — `1.0` se query e candidato
  condividono un'entita' (sono a 2 salti nel grafo), altrimenti `0.0`;
- **importanza composita del candidato** (Lezione 25) — quanto quel
  candidato conta *a prescindere* dalla query (esplicito + recency +
  type).

Un punteggio ibrido, di nuovo con pesi **dichiarati** (lo stesso principio
della Lezione 25):

```
ibrido(query, candidato) = w1*similarita + w2*segnale_grafo + w3*importanza_candidato
```

qui `w1=0.5, w2=0.3, w3=0.2` — la similarita' resta il segnale principale,
il grafo e l'importanza lo correggono senza sovrastarlo.

**Cosa NON aspettarti**: che il retrieval ibrido vinca su **ogni**
metrica della Lezione 21 rispetto al solo embedding. Il proxy di
rilevanza usato li' (stesso `type`) e' dichiaratamente imperfetto (Lezione
21: "due memorie di type diverso potrebbero comunque essere correlate").
Il segnale di grafo ottimizza per una nozione di rilevanza **diversa**
(stessa entita' esplicita, non stesso type): puo' promuovere in cima un
candidato che condivide una persona o un luogo con la query, anche se il
suo `type` e' diverso — un risultato arguibilmente piu' utile in pratica,
ma "sbagliato" secondo la metrica basata sul type. Vedrai questo esatto
caso nella Parte 3.

In [1]:
W1, W2, W3 = 0.5, 0.3, 0.2
assert abs(W1 + W2 + W3 - 1.0) < 1e-9


def punteggio_ibrido(similarita, segnale_grafo, importanza_candidato):
    return W1 * similarita + W2 * segnale_grafo + W3 * importanza_candidato


# Due candidati con la stessa similarita' ma segnali diversi.
candidato_con_grafo = punteggio_ibrido(similarita=0.90, segnale_grafo=1.0, importanza_candidato=0.5)
candidato_senza_grafo = punteggio_ibrido(similarita=0.90, segnale_grafo=0.0, importanza_candidato=0.9)
print(f'candidato collegato via grafo    : {candidato_con_grafo:.3f}')
print(f'candidato non collegato, piu\' importante: {candidato_senza_grafo:.3f}')

candidato collegato via grafo    : 0.850
candidato non collegato, piu' importante: 0.630


Leggi l'output: a parita' di similarita' (`0.90` per entrambi), il
segnale di grafo (`w2=0.3`) puo' ribaltare l'ordine rispetto a una
differenza di importanza (`w3=0.2`) — qui il candidato collegato via
grafo vince nonostante un'importanza composita piu' bassa, perche' il suo
peso (`0.3`) e' maggiore di quello dell'importanza (`0.2`). Non e' un
caso: riflette la scelta dichiarata di pesare il grafo piu' della sola
importanza del candidato.

## Parte 2 — Esercizio guidato

Il tuo compito: calcola `punteggio_ibrido` per un candidato con
`similarita=0.5`, `segnale_grafo=0.0`, `importanza_candidato=1.0` (il
massimo possibile per l'importanza, Lezione 25) e confrontalo con un
candidato `similarita=0.5, segnale_grafo=1.0, importanza_candidato=0.0`.
Quale vince, e perche'?

In [2]:
# Scrivi qui: calcola i due punteggi e stampali.

pass

### Soluzione spiegata

`punteggio_ibrido(0.5, 0.0, 1.0) = 0.5*0.5 + 0.3*0.0 + 0.2*1.0 = 0.45`.
`punteggio_ibrido(0.5, 1.0, 0.0) = 0.5*0.5 + 0.3*1.0 + 0.2*0.0 = 0.55`.
Vince il secondo: il peso del segnale di grafo (`0.3`) e' maggiore di
quello dell'importanza (`0.2`), quindi un candidato collegato via grafo
con importanza zero batte un candidato isolato con importanza massima —
di nuovo, conseguenza diretta dei pesi dichiarati, non un caso.

In [3]:
a = punteggio_ibrido(similarita=0.5, segnale_grafo=0.0, importanza_candidato=1.0)
b = punteggio_ibrido(similarita=0.5, segnale_grafo=1.0, importanza_candidato=0.0)
print(f'candidato A (solo importanza alta): {a:.3f}')
print(f'candidato B (solo segnale grafo)  : {b:.3f}')
assert b > a

candidato A (solo importanza alta): 0.450
candidato B (solo segnale grafo)  : 0.550


## Parte 3 — Il progetto: Memory AI Lab, passo 28 — confronto onesto

Ricostruiamo embedding (Lezioni 17-18), grafo (Lezione 27) e importanza
composita (Lezione 25) sulle memorie di validation, poi confrontiamo
retrieval **solo embedding** contro **ibrido** con le metriche della
Lezione 21 (Precision@3, Recall@3, MRR, proxy di rilevanza = stesso
type).

In [4]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import re
import networkx as nx
import pandas as pd
import keras
from keras.layers import TextVectorization
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

keras.utils.set_random_seed(42)

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

LUNGHEZZA_SEQUENZA = 24
vettorizzatore = TextVectorization(max_tokens=300, output_mode='int',
                                   output_sequence_length=LUNGHEZZA_SEQUENZA)
vettorizzatore.adapt(testi['train'])
X_seq = {n: vettorizzatore(t).numpy() for n, t in testi.items()}

ingresso = keras.Input(shape=(LUNGHEZZA_SEQUENZA,))
parole = keras.layers.Embedding(input_dim=300, output_dim=16, mask_zero=True)(ingresso)
vettore_frase = keras.layers.GlobalAveragePooling1D(name='sentence_embedding')(parole)
nascosto = keras.layers.Dense(32, activation='relu')(vettore_frase)
nascosto = keras.layers.Dropout(0.3)(nascosto)
uscita = keras.layers.Dense(4, activation='softmax')(nascosto)

classificatore = keras.Model(ingresso, uscita)
classificatore.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                       metrics=['accuracy'])
classificatore.fit(X_seq['train'], target['train'], epochs=300,
                   validation_data=(X_seq['val'], target['val']),
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                            restore_best_weights=True)],
                   verbose=0)

incorporatore = keras.Model(classificatore.input,
                            classificatore.get_layer('sentence_embedding').output)
vettori_val = incorporatore(X_seq['val']).numpy()
tipi_val = memorie['val']['type'].to_numpy()
n = len(tipi_val)
matrice_similarita = cosine_similarity(vettori_val)
print('embedding pronti:', vettori_val.shape)

2026-07-18 10:21:29.555476: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


embedding pronti: (20, 16)


In [5]:
val = memorie['val'].reset_index(drop=True)

STOPWORD_MAIUSCOLE = {'The'}


def estrai_entita(testo):
    candidate = re.findall(r'\b[A-Z][a-zA-Z]*\b', str(testo))
    return [c for c in candidate if c not in STOPWORD_MAIUSCOLE]


val['entita'] = val['text'].apply(estrai_entita)

G = nx.Graph()
for _, riga in val.iterrows():
    G.add_node(riga['memory_id'], tipo='memoria')
    for entita in riga['entita']:
        G.add_node(entita, tipo='entita')
        G.add_edge(riga['memory_id'], entita)


def memorie_correlate(memory_id):
    correlate = set()
    for entita in G.neighbors(memory_id):
        for m in G.neighbors(entita):
            if m != memory_id:
                correlate.add(m)
    return correlate


ORA_RIFERIMENTO = pd.Timestamp('2026-07-18')
PARAMETRI_TIPO = {
    'episodic':   {'half_life_giorni': 30,  'peso_importanza_base': 1.0},
    'semantic':   {'half_life_giorni': 365, 'peso_importanza_base': 1.1},
    'preference': {'half_life_giorni': 180, 'peso_importanza_base': 1.2},
    'unknown':    {'half_life_giorni': 60,  'peso_importanza_base': 0.7},
}
eta_giorni = (ORA_RIFERIMENTO - pd.to_datetime(val['timestamp'])).dt.total_seconds() / 86400
half_life = val['type'].map(lambda t: PARAMETRI_TIPO[t]['half_life_giorni'])
peso_base = val['type'].map(lambda t: PARAMETRI_TIPO[t]['peso_importanza_base'])
recency = 0.5 ** (eta_giorni / half_life)
ALPHA, BETA = 0.6, 0.4
val['importanza_composita'] = peso_base * (ALPHA * val['importance'] + BETA * recency)

ids = val['memory_id'].to_numpy()


def punteggio_ibrido_coppia(i, j):
    sim = matrice_similarita[i, j]
    segnale_grafo = 1.0 if ids[j] in memorie_correlate(ids[i]) else 0.0
    importanza = val.loc[j, 'importanza_composita']
    return punteggio_ibrido(sim, segnale_grafo, importanza)

In [6]:
import numpy as np

K = 3


def valuta_retrieval(punteggio_fn):
    precisioni, recall_lista, reciprocal_ranks = [], [], []
    for i in range(n):
        rilevanti_totali = np.sum(tipi_val == tipi_val[i]) - 1
        if rilevanti_totali == 0:
            continue
        ranking = sorted(((j, punteggio_fn(i, j)) for j in range(n) if j != i),
                         key=lambda coppia: -coppia[1])
        topk = [j for j, _ in ranking[:K]]
        rilevanti_in_topk = sum(1 for j in topk if tipi_val[j] == tipi_val[i])
        precisioni.append(rilevanti_in_topk / K)
        recall_lista.append(rilevanti_in_topk / rilevanti_totali)
        for rank, (j, _) in enumerate(ranking, start=1):
            if tipi_val[j] == tipi_val[i]:
                reciprocal_ranks.append(1 / rank)
                break
    return np.mean(precisioni), np.mean(recall_lista), np.mean(reciprocal_ranks), len(precisioni)


p_base, r_base, mrr_base, nq = valuta_retrieval(lambda i, j: matrice_similarita[i, j])
p_ibr, r_ibr, mrr_ibr, _ = valuta_retrieval(punteggio_ibrido_coppia)

print(f'{"metrica":12s}{"solo embedding":>16s}{"ibrido":>12s}')
print(f'{"Precision@3":12s}{p_base:>16.3f}{p_ibr:>12.3f}')
print(f'{"Recall@3":12s}{r_base:>16.3f}{r_ibr:>12.3f}')
print(f'{"MRR":12s}{mrr_base:>16.3f}{mrr_ibr:>12.3f}')
print(f'\n(query valutate: {nq}/{n})')

metrica       solo embedding      ibrido
Precision@3            0.907       0.963
Recall@3               0.319       0.361
MRR                    1.000       0.972

(query valutate: 18/20)


Guarda la tabella **senza cercare un vincitore assoluto**: nell'esecuzione
di riferimento l'ibrido migliora Precision@3 e Recall@3, ma **peggiora**
leggermente l'MRR (da 1.000 a un valore piu' basso) — esattamente
l'avviso della Parte 1. Guardiamo dentro a quella singola query dove
succede, per capire *perche'*, non solo *che* succede.

In [7]:
for i in range(n):
    rilevanti_totali = np.sum(tipi_val == tipi_val[i]) - 1
    if rilevanti_totali == 0:
        continue
    ranking_base = sorted(((j, matrice_similarita[i, j]) for j in range(n) if j != i),
                          key=lambda c: -c[1])
    primo_rilevante_base = next(rank for rank, (j, _) in enumerate(ranking_base, start=1)
                                if tipi_val[j] == tipi_val[i])
    ranking_ibrido = sorted(((j, punteggio_ibrido_coppia(i, j)) for j in range(n) if j != i),
                            key=lambda c: -c[1])
    primo_rilevante_ibrido = next(rank for rank, (j, _) in enumerate(ranking_ibrido, start=1)
                                  if tipi_val[j] == tipi_val[i])
    if primo_rilevante_base != primo_rilevante_ibrido:
        print(f"query: {val.loc[i, 'memory_id']} [{tipi_val[i]}] \"{val.loc[i, 'text']}\"")
        print(f'  primo risultato rilevante (per type) -> baseline: posizione {primo_rilevante_base}, '
              f'ibrido: posizione {primo_rilevante_ibrido}')
        print(f"  primo posto ibrido: {val.loc[ranking_ibrido[0][0], 'memory_id']} "
              f"[{tipi_val[ranking_ibrido[0][0]]}] \"{val.loc[ranking_ibrido[0][0], 'text']}\" "
              f"(entita' condivise: {set(val.loc[i, 'entita']) & set(val.loc[ranking_ibrido[0][0], 'entita'])})")

query: hist_0215 [semantic] "Lucia works on la riunione settimanale."
  primo risultato rilevante (per type) -> baseline: posizione 1, ibrido: posizione 2
  primo posto ibrido: hist_0250 [unknown] "Lucia works on il colloquio." (entita' condivise: {'Lucia'})


Ecco il caso concreto: per la query *"Lucia works on la riunione
settimanale."* (`semantic`), il solo embedding mette al primo posto una
memoria dello stesso type ("Sara works on..."), quindi l'MRR e' perfetto
secondo il proxy "stesso type". L'ibrido promuove invece in cima "Lucia
works on il colloquio." — type `unknown`, **ma la stessa persona
(`Lucia`)** della query. Secondo il proxy della Lezione 21 questo conta
come un "errore" (MRR peggiore), ma nella pratica una memoria sulla
**stessa persona** e' un risultato ragionevole, forse persino piu' utile,
di una memoria di tipo uguale ma su una persona diversa (`Sara`). Questo
e' esattamente il limite del proxy di rilevanza gia' dichiarato nella
Lezione 21: "stesso type" non e' l'unica nozione valida di rilevanza, e un
segnale come il grafo puo' ottimizzare per una nozione diversa, non
peggiore, solo non misurata dalla stessa metrica.

## Cosa hai imparato

- Un **punteggio ibrido** combina segnali con pesi dichiarati, esattamente
  come l'importanza composita della Lezione 25 — non c'e' modo di evitare
  di dichiarare quanto ciascun segnale conta.
- Un miglioramento in una metrica (Precision@3, Recall@3) puo'
  accompagnarsi a un peggioramento in un'altra (MRR): **non e' un bug**,
  e' il segno che i segnali combinati ottimizzano per nozioni di
  rilevanza leggermente diverse.
- Il proxy "stesso type" (Lezione 21) e' onestamente imperfetto: un
  risultato "sbagliato" secondo quella metrica (type diverso) puo' essere
  comunque utile in pratica (stessa entita' condivisa) — leggere solo il
  numero aggregato senza guardare dentro ai casi concreti nasconde
  proprio questa sfumatura.

## Quiz

1. Perche' non e' garantito che il retrieval ibrido migliori **tutte** le
   metriche di retrieval contemporaneamente rispetto al solo embedding?
2. Nell'esempio concreto della Parte 3, perche' il risultato promosso
   dall'ibrido non e' necessariamente "peggiore" nella pratica, anche se
   l'MRR basato sul type scende?
3. Se volessi che il grafo pesasse meno nel punteggio ibrido, quale
   parametro cambieresti, e cosa ti aspetteresti che succedesse ai
   risultati della Parte 3?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' i segnali combinati (similarita', grafo, importanza) misurano
   aspetti diversi di "rilevanza", e la metrica usata per valutare
   (proxy "stesso type") ne cattura solo uno parzialmente. Ottimizzare
   una combinazione di segnali puo' migliorare l'allineamento medio con
   il type in alcuni casi (Precision@3, Recall@3) e peggiorarlo in altri
   (il primo risultato non e' piu' dello stesso type, MRR scende).
2. Perche' il proxy "stesso type" e' una semplificazione dichiarata fin
   dalla Lezione 21, non una definizione esaustiva di rilevanza: una
   memoria sulla stessa persona (`Lucia`) puo' essere piu' utile in
   pratica di una memoria dello stesso `type` ma su una persona diversa,
   anche se la metrica basata sul type la conta come "sbagliata".
3. Ridurresti `w2` (il peso del segnale di grafo) e aumenteresti
   corrispondentemente `w1` o `w3` per mantenere la somma a 1: con `w2`
   piu' basso, il grafo influenzerebbe meno l'ordinamento finale, e i
   risultati dovrebbero avvicinarsi a quelli del solo embedding (incluso
   l'MRR, che tornerebbe verso 1.000).
</details>

## Fonti

Lezione di sintesi: combina segnali e metriche gia' documentati nelle
Lezioni 18, 21, 25 e 27. Nessuna nuova API esterna, rilistate qui per
comodita':

- scikit-learn, `cosine_similarity`:
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- NetworkX, `Graph`:
  https://networkx.org/documentation/stable/reference/classes/graph.html

## Prossima lezione

Il retrieval ibrido assume che le memorie siano tutte valide
contemporaneamente. Non e' sempre vero: l'ultima lezione della Fase 4
affronta le **contraddizioni** — quando una nuova memoria smentisce una
vecchia, e come aggiornare l'archivio di conseguenza.